<a href="https://www.kaggle.com/code/ebubekirtilbac/flo-rfm-analysis?scriptVersionId=226811264" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/flo-crm-data/flo_data_20k.csv


**FLO CRM DATASET**

![FLO CRM DATASET](https://www.mngavm.com/wp-content/uploads/2022/12/166b4b9a6db1bc18d0535864982ab055.png)

In [2]:
#DATA UNDERSTANDING AND PREPARING
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 500)
df_=pd.read_csv(r"/kaggle/input/flo-crm-data/flo_data_20k.csv")
df=df_.copy()

In [3]:
###DATA CHECKING####

def df_check(df, head_rows=5):
    print(df.shape)
    print(df.dtypes)
    print(df.head(head_rows))
    print(df.tail(head_rows))
    print(df.describe().T)
    print(df.isnull().sum())
    print(df.memory_usage(deep=True))

df_check(df)

(19945, 12)
master_id                             object
order_channel                         object
last_order_channel                    object
first_order_date                      object
last_order_date                       object
last_order_date_online                object
last_order_date_offline               object
order_num_total_ever_online          float64
order_num_total_ever_offline         float64
customer_value_total_ever_offline    float64
customer_value_total_ever_online     float64
interested_in_categories_12           object
dtype: object
                              master_id order_channel last_order_channel first_order_date last_order_date last_order_date_online last_order_date_offline  order_num_total_ever_online  order_num_total_ever_offline  customer_value_total_ever_offline  customer_value_total_ever_online       interested_in_categories_12
0  cc294636-19f0-11eb-8d74-000d3a38a36f   Android App            Offline       2020-10-30      2021-02-26             2

*****Let’s create new variables for each customer’s total number of purchases and total spending.** ****![](https://www.zeneks.com.tr/wp-content/uploads/2022/11/KURUMSAL-CRM2.png)

In [4]:
df["total_order_num"] = df["order_num_total_ever_offline"] + df["order_num_total_ever_online"]
df["total_customer_value"] = df["customer_value_total_ever_offline"] + df["customer_value_total_ever_online"]

In [5]:
for col in df.columns:
    if "date" in col:
        print(col)
        df[col] = pd.to_datetime(df[col])

df.info()

first_order_date
last_order_date
last_order_date_online
last_order_date_offline
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19945 entries, 0 to 19944
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   master_id                          19945 non-null  object        
 1   order_channel                      19945 non-null  object        
 2   last_order_channel                 19945 non-null  object        
 3   first_order_date                   19945 non-null  datetime64[ns]
 4   last_order_date                    19945 non-null  datetime64[ns]
 5   last_order_date_online             19945 non-null  datetime64[ns]
 6   last_order_date_offline            19945 non-null  datetime64[ns]
 7   order_num_total_ever_online        19945 non-null  float64       
 8   order_num_total_ever_offline       19945 non-null  float64       
 9   customer_value_total_ever

*****Let’s examine the distribution of the number of customers, the average number of products purchased, and the average spending across shopping channels.*****

In [6]:
df.groupby("order_channel").agg({"total_customer_value":"mean", "total_order_num": "mean"})

,total_customer_value,total_order_num
order_channel,,
Android App,823.492655,5.504897
Desktop,588.782984,3.992687
Ios App,891.634285,5.418637
Mobile,620.275125,4.440598


***Let’s rank the top 10 customers generating the highest revenue.***

In [7]:
df[["master_id", "total_customer_value"]].sort_values("total_customer_value", ascending=False).head(10)

,master_id,total_customer_value
11150,5d1c466a-9cfd-11e9-9897-000d3a38a36f,45905.10
4315,d5ef8058-a5c6-11e9-a2fc-000d3a38a36f,36818.29
7613,73fd19aa-9e37-11e9-9897-000d3a38a36f,33918.10
13880,7137a5c0-7aad-11ea-8f20-000d3a38a36f,31227.41
9055,47a642fe-975b-11eb-8c2a-000d3a38a36f,20706.34
7330,a4d534a2-5b1b-11eb-8dbd-000d3a38a36f,18443.57
8068,d696c654-2633-11ea-8e1c-000d3a38a36f,16918.57
163,fef57ffa-aae6-11e9-a2fc-000d3a38a36f,12726.10
7223,cba59206-9dd1-11e9-9897-000d3a38a36f,12282.24
18767,fc0ce7a4-9d87-11e9-9897-000d3a38a36f,12103.15


***Let’s rank the top 10 customers with the highest number of orders.***

In [8]:
df[["master_id", "total_order_num"]].sort_values("total_order_num", ascending=False).head(10)

,master_id,total_order_num
11150,5d1c466a-9cfd-11e9-9897-000d3a38a36f,202.0
7223,cba59206-9dd1-11e9-9897-000d3a38a36f,131.0
8783,a57f4302-b1a8-11e9-89fa-000d3a38a36f,111.0
2619,fdbe8304-a7ab-11e9-a2fc-000d3a38a36f,88.0
6322,329968c6-a0e2-11e9-a2fc-000d3a38a36f,83.0
7613,73fd19aa-9e37-11e9-9897-000d3a38a36f,82.0
9347,44d032ee-a0d4-11e9-a2fc-000d3a38a36f,77.0
10954,b27e241a-a901-11e9-a2fc-000d3a38a36f,75.0
8068,d696c654-2633-11ea-8e1c-000d3a38a36f,70.0
7330,a4d534a2-5b1b-11eb-8dbd-000d3a38a36f,70.0


**LET’S CALCULATE THE RFM METRICS.**

***RFM:“RFM (Recency, Frequency, Monetary) is a customer segmentation technique that analyzes how recently, how often, and how much a customer spends.”***


![](http://analyticahouse.com/Website/assets/img/Blogs/6314b7a379496.png)

In [9]:
import datetime as dt
max_date = df["last_order_date"].max()
today_date = max_date + dt.timedelta(days=2)

rfm = df.groupby("master_id").agg({"last_order_date": lambda date: (today_date - date.max()).days,
                             "total_order_num": lambda order: order.sum(),
                             "total_customer_value": lambda value: value.sum()})
rfm.columns = ["recency", "frequency", "monetary"]
rfm.head(10)

,recency,frequency,monetary
master_id,,,
00016786-2f5a-11ea-bb80-000d3a38a36f,10,5.0,776.07
00034aaa-a838-11e9-a2fc-000d3a38a36f,298,3.0,269.47
000be838-85df-11ea-a90b-000d3a38a36f,213,4.0,722.69
000c1fe2-a8b7-11ea-8479-000d3a38a36f,27,7.0,874.16
000f5e3e-9dde-11ea-80cd-000d3a38a36f,20,7.0,1620.33
00136ce2-a562-11e9-a2fc-000d3a38a36f,203,2.0,359.45
00142f9a-7af6-11eb-8460-000d3a38a36f,25,3.0,404.94
0014778a-5b11-11ea-9a2c-000d3a38a36f,26,3.0,727.43
0018c6aa-ab6c-11e9-a2fc-000d3a38a36f,126,2.0,317.91


In [10]:
##Calculation of RF and RFM Scores

##RF Score (Recency & Frequency): A combined metric that evaluates how recently and how often a customer makes purchases.
##	RFM Score (Recency, Frequency, Monetary): A customer segmentation model that considers how recently a customer purchased, how often they buy, and how much they spend.


rfm["recency_score"] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])
rfm["frequency_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
rfm["monetary_score"] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])
rfm["RF_SCORE"] = (rfm['recency_score'].astype(str) +
                    rfm['frequency_score'].astype(str))

rfm.head()

,recency,frequency,monetary,recency_score,frequency_score,monetary_score,RF_SCORE
master_id,,,,,,,
00016786-2f5a-11ea-bb80-000d3a38a36f,10,5.0,776.07,5,4,4,54
00034aaa-a838-11e9-a2fc-000d3a38a36f,298,3.0,269.47,1,2,1,12
000be838-85df-11ea-a90b-000d3a38a36f,213,4.0,722.69,2,3,4,23
000c1fe2-a8b7-11ea-8479-000d3a38a36f,27,7.0,874.16,5,4,4,54
000f5e3e-9dde-11ea-80cd-000d3a38a36f,20,7.0,1620.33,5,4,5,54


***Let’s define RF scores as segments***

In [11]:
segment_map = {
    r'[1-2][1-2]': 'hibernating',
    r'[1-2][3-4]': 'at_risk',
    r'[1-2]5': 'cant_loose',
    r'3[1-2]': 'about_to_sleep',
    r'33': 'need_attention',
    r'[3-4][4-5]': 'loyal_customers',
    r'41': 'promising',
    r'51': 'new_customers',
    r'[4-5][2-3]': 'potential_loyalists',
    r'5[4-5]': 'champions'
}

rfm['segment'] = rfm['RF_SCORE'].replace(segment_map, regex=True)

rfm.head()

,recency,frequency,monetary,recency_score,frequency_score,monetary_score,RF_SCORE,segment
master_id,,,,,,,,
00016786-2f5a-11ea-bb80-000d3a38a36f,10,5.0,776.07,5,4,4,54,champions
00034aaa-a838-11e9-a2fc-000d3a38a36f,298,3.0,269.47,1,2,1,12,hibernating
000be838-85df-11ea-a90b-000d3a38a36f,213,4.0,722.69,2,3,4,23,at_risk
000c1fe2-a8b7-11ea-8479-000d3a38a36f,27,7.0,874.16,5,4,4,54,champions
000f5e3e-9dde-11ea-80cd-000d3a38a36f,20,7.0,1620.33,5,4,5,54,champions


***Let’s analyze the average recency, frequency, and monetary values of the segments.***

In [12]:
rfm[["segment", "recency", "frequency", "monetary"]].groupby("segment").agg(["mean", "count"])

recency        frequency           monetary      
                           mean count       mean count         mean count
segment                                                                  
about_to_sleep       114.031649  1643   2.406573  1643   361.649373  1643
at_risk              242.328997  3152   4.470178  3152   648.325038  3152
cant_loose           235.159129  1194  10.716918  1194  1481.652446  1194
champions             17.142187  1920   8.965104  1920  1410.708938  1920
hibernating          247.426303  3589   2.391474  3589   362.583299  3589
loyal_customers       82.557926  3375   8.356444  3375  1216.257224  3375
need_attention       113.037221   806   3.739454   806   553.436638   806
new_customers         17.976226   673   2.000000   673   344.049495   673
potential_loyalists   36.869744  2925   3.310769  2925   533.741344  2925
promising             58.694611   668   2.000000   668   334.153338   668

**Mission time**




**Case 1: FLO is adding a new women’s shoe brand to its portfolio. The prices of this brand’s products are above the general customer preferences. Therefore, they want to communicate specifically with customers who match the target profile for brand promotion and product sales. Loyal customers (champions, loyal_customers) who have spent over 250 TL on average and have purchased from the women’s category will be contacted. Save the master_id numbers of these customers in a CSV file named new_brand_target_customer_id.csv.**


In [13]:
combined = pd.merge(df, rfm, on="master_id")

combined[((combined["segment"] == "champions") | 
          (combined["segment"] == "loyal_customers")) & 
         ((combined["total_customer_value"] / combined["total_order_num"]) > 250) & 
         (combined["interested_in_categories_12"].str.contains("KADIN"))]["master_id"]

74       8423eda8-ae37-11e9-a2fc-000d3a38a36f
136      37753730-9b69-11ea-a27a-000d3a38a36f
1078     df760a84-ab25-11e9-a2fc-000d3a38a36f
1134     fdd08240-244a-11ea-a30f-000d3a38a36f
1323     1df22bd2-8549-11ea-80af-000d3a38a36f
1484     aa100b7a-d667-11e9-93bc-000d3a38a36f
1574     f2affcfc-f186-11e9-9346-000d3a38a36f
1682     667a10cc-4fa2-11ea-99ba-000d3a38a36f
2002     f1e095a6-637e-11ea-a6dc-000d3a38a36f
2060     3a2a6c48-526c-11ea-956d-000d3a38a36f
2246     169d1518-9d6e-11e9-9897-000d3a38a36f
2401     1e4fcad2-a57d-11e9-a2fc-000d3a38a36f
2542     70d34b80-a7e4-11e9-a2fc-000d3a38a36f
2763     215e3162-565a-11ea-b9b9-000d3a38a36f
3157     25a05fe6-5a84-11eb-9e65-000d3a38a36f
3274     c2984396-d622-11e9-93bc-000d3a38a36f
3662     63caf5e4-e3be-11e9-ad00-000d3a38a36f
3879     b43a543e-5b96-11ea-a3e9-000d3a38a36f
3932     f8033de8-b16e-11e9-89fa-000d3a38a36f
3996     715f95a2-9f94-11e9-a2fc-000d3a38a36f
4016     b95511d4-aa3e-11e9-a2fc-000d3a38a36f
4130     ea56ef18-b1a1-11e9-89fa-0

In [14]:
combined[((combined["segment"] == "champions") | 
          (combined["segment"] == "loyal_customers")) & 
         ((combined["total_customer_value"] / combined["total_order_num"]) > 250) & 
         (combined["interested_in_categories_12"].str.contains("KADIN"))]["master_id"].to_csv("new_brand_target_customer_id.csv")
print(pd.read_csv("new_brand_target_customer_id.csv").head(20))

    Unnamed: 0                             master_id
0           74  8423eda8-ae37-11e9-a2fc-000d3a38a36f
1          136  37753730-9b69-11ea-a27a-000d3a38a36f
2         1078  df760a84-ab25-11e9-a2fc-000d3a38a36f
3         1134  fdd08240-244a-11ea-a30f-000d3a38a36f
4         1323  1df22bd2-8549-11ea-80af-000d3a38a36f
5         1484  aa100b7a-d667-11e9-93bc-000d3a38a36f
6         1574  f2affcfc-f186-11e9-9346-000d3a38a36f
7         1682  667a10cc-4fa2-11ea-99ba-000d3a38a36f
8         2002  f1e095a6-637e-11ea-a6dc-000d3a38a36f
9         2060  3a2a6c48-526c-11ea-956d-000d3a38a36f
10        2246  169d1518-9d6e-11e9-9897-000d3a38a36f
11        2401  1e4fcad2-a57d-11e9-a2fc-000d3a38a36f
12        2542  70d34b80-a7e4-11e9-a2fc-000d3a38a36f
13        2763  215e3162-565a-11ea-b9b9-000d3a38a36f
14        3157  25a05fe6-5a84-11eb-9e65-000d3a38a36f
15        3274  c2984396-d622-11e9-93bc-000d3a38a36f
16        3662  63caf5e4-e3be-11e9-ad00-000d3a38a36f
17        3879  b43a543e-5b96-11ea-a3e9-000d3a

**Case 2:
A discount of approximately 40% is planned for men’s and children’s products. The goal is to specifically target past high-value customers who have not shopped for a long time, dormant customers, and new customers who should not be lost but are interested in these categories. Save the IDs of the suitable target customers in a CSV file named discount_target_customer_ids.csv.**

In [15]:
combined[((combined["segment"] == "hibernating") | 
          (combined["segment"] == "cant_loose") | 
          (combined["segment"] == "new_customers")) & 
         ((combined["interested_in_categories_12"].str.contains("ERKEK")) | 
          (combined["interested_in_categories_12"].str.contains("COCUK")))]["master_id"]

7        3f1b4dc8-8a7d-11ea-8ec0-000d3a38a36f
10       ae608ece-c9d8-11ea-a31e-000d3a38a36f
15       13ed97a4-b167-11e9-89fa-000d3a38a36f
19       2730793e-3908-11ea-85d6-000d3a38a36f
21       7b289956-d691-11e9-93bc-000d3a38a36f
29       a46cc34a-a19f-11e9-a2fc-000d3a38a36f
35       7a52fd46-5be8-11ea-a7c9-000d3a38a36f
48       1eb32dea-a409-11e9-a2fc-000d3a38a36f
66       bb548d60-abb6-11e9-a2fc-000d3a38a36f
76       7d58deb6-62fa-11ea-a6dc-000d3a38a36f
91       808005da-a511-11e9-a2fc-000d3a38a36f
108      5bc87a7c-9de8-11e9-9897-000d3a38a36f
122      f318a3de-ad06-11e9-a2fc-000d3a38a36f
130      ae557c64-5b29-11ea-b304-000d3a38a36f
145      5bfa131c-ada5-11e9-a2fc-000d3a38a36f
147      90e0b158-2b54-11ea-9d27-000d3a38a36f
148      ecabc266-a7f6-11e9-a2fc-000d3a38a36f
149      0117c094-63e8-11ea-a6dc-000d3a38a36f
156      c88819fa-d0fb-11ea-8b59-000d3a38a36f
166      9613613c-c9d0-11ea-a31e-000d3a38a36f
170      c25f2eea-265d-11ea-b9a0-000d3a38a36f
175      737ef46a-18eb-11ea-9213-0

In [16]:
combined[((combined["segment"] == "hibernating") | 
          (combined["segment"] == "cant_loose") | 
          (combined["segment"] == "new_customers")) & 
         ((combined["interested_in_categories_12"].str.contains("ERKEK")) | 
          (combined["interested_in_categories_12"].str.contains("COCUK")))]["master_id"].to_csv("discount_target_customer_ids.csv")
print(pd.read_csv("discount_target_customer_ids.csv").head(20))

    Unnamed: 0                             master_id
0            7  3f1b4dc8-8a7d-11ea-8ec0-000d3a38a36f
1           10  ae608ece-c9d8-11ea-a31e-000d3a38a36f
2           15  13ed97a4-b167-11e9-89fa-000d3a38a36f
3           19  2730793e-3908-11ea-85d6-000d3a38a36f
4           21  7b289956-d691-11e9-93bc-000d3a38a36f
5           29  a46cc34a-a19f-11e9-a2fc-000d3a38a36f
6           35  7a52fd46-5be8-11ea-a7c9-000d3a38a36f
7           48  1eb32dea-a409-11e9-a2fc-000d3a38a36f
8           66  bb548d60-abb6-11e9-a2fc-000d3a38a36f
9           76  7d58deb6-62fa-11ea-a6dc-000d3a38a36f
10          91  808005da-a511-11e9-a2fc-000d3a38a36f
11         108  5bc87a7c-9de8-11e9-9897-000d3a38a36f
12         122  f318a3de-ad06-11e9-a2fc-000d3a38a36f
13         130  ae557c64-5b29-11ea-b304-000d3a38a36f
14         145  5bfa131c-ada5-11e9-a2fc-000d3a38a36f
15         147  90e0b158-2b54-11ea-9d27-000d3a38a36f
16         148  ecabc266-a7f6-11e9-a2fc-000d3a38a36f
17         149  0117c094-63e8-11ea-a6dc-000d3a

***“Thank you for reviewing!” 😊***


***EBUBEKİR TİLBAÇ***


![](https://www.icegif.com/wp-content/uploads/2024/12/thank-you-icegif-7.gif)